# Recovered ZuCo: secondary small CNN
Run the [CPU diagnostics/classification notebook](https://colab.research.google.com/github/parmisbathayan/word-sentiment-eeg/blob/main/notebooks/zuco_recovered_followups.ipynb?authuser=2) first. Then select **T4 GPU** here and **Run all**, using the same Zyion Drive account.

Independent compact EEGNet-inspired model; fixed settings, individual fixation 0–800 ms input, raw/clean, unseen sentences/words/readers. Inner grouped validation selects the epoch; test data never select settings. At most 25 epochs/fold, early stopping. No model weights are downloaded.

Loads existing saved trials; no native extraction or cleaning. New files share the same **50 MB total** follow-up cap. Completed folds and small model frames persist on Drive; only unfinished work repeats after a disconnect. No EEG/features are duplicated on Drive. GPU results may differ slightly across hardware/library versions.


In [ ]:
# Runtime check: run before installation, Drive access, or analysis.
import os
import subprocess

REQUIRED_RUNTIME = 'GPU'
if os.environ.get('COLAB_TPU_ADDR') or os.environ.get('TPU_NAME'):
    raise RuntimeError(f'This notebook needs {REQUIRED_RUNTIME}, not TPU. Choose Runtime > Change runtime type, then rerun all.')
try:
    _gpu_probe = subprocess.run(
        ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=15, check=False)
except FileNotFoundError:
    _gpu_names = ''
except subprocess.TimeoutExpired:
    raise RuntimeError('GPU detection timed out. Reconnect the runtime and rerun this cell.') from None
else:
    if _gpu_probe.returncode:
        raise RuntimeError('GPU detection failed. Reconnect the runtime and rerun this cell.')
    _gpu_names = _gpu_probe.stdout.strip()
if REQUIRED_RUNTIME == 'GPU' and not _gpu_names:
    raise RuntimeError('GPU required. Choose Runtime > Change runtime type > T4 GPU, then Run all. No setup or analysis has started.')
if REQUIRED_RUNTIME == 'CPU' and _gpu_names:
    raise RuntimeError('CPU notebook: choose Runtime > Change runtime type > CPU (or hardware accelerator None), then Run all to avoid using GPU quota.')
print('Runtime verified:', _gpu_names if _gpu_names else 'CPU', flush=True)


In [ ]:
from google.colab import drive, auth
from pathlib import Path
import os, sys, subprocess, json
from IPython.display import display, Markdown, Image
import pandas as pd

drive.mount('/content/drive')
THESIS=Path('/content/drive/MyDrive/Thesis')
RUN=THESIS/'Results/word_sentiment_eeg/zuco_paper_figures_recovered'
OUT=THESIS/'Results/word_sentiment_eeg/zuco_recovered_followups'
REPO=Path('/content/word-sentiment-eeg-recovered-followups')
CODE_REVISION='6a4624e40725740e8bde7d8e993023477267b5ff'
def git(*args):
    result=subprocess.run(['git','-c','credential.helper=','-c','http.extraheader=',*map(str,args)],
                          text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                          env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
    if result.returncode:
        raise RuntimeError('Git setup failed:\n'+result.stdout)
    if result.stdout:print(result.stdout,end='',flush=True)
if not REPO.exists():
    git('clone','--depth','1','https://github.com/parmisbathayan/word-sentiment-eeg.git',REPO)
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary checkout contains edits. Preserve them before updating.')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    git('-C',REPO,'fetch','--depth','1','origin',CODE_REVISION)
    fetched=subprocess.check_output(['git','-C',str(REPO),'rev-parse','FETCH_HEAD^{commit}'],text=True).strip()
    if fetched!=CODE_REVISION:raise RuntimeError('Fetched code differs from the pinned commit.')
    git('-C',REPO,'checkout','--detach','FETCH_HEAD')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    raise RuntimeError('Code checkout verification failed.')
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements-paper.txt')],check=True)
print('Code revision:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())

# A mounted Drive filesystem disk_usage value can reflect the runtime volume;
# use the authenticated Drive quota instead.
auth.authenticate_user()
import google.auth
from google.auth.transport.requests import AuthorizedSession
credentials,_=google.auth.default()
response=AuthorizedSession(credentials).get('https://www.googleapis.com/drive/v3/about',
    params={'fields':'storageQuota,user(emailAddress)'},timeout=30)
response.raise_for_status();quota=response.json()['storageQuota']
print('Quota account:',response.json().get('user',{}).get('emailAddress','unavailable'))
if 'limit' not in quota:
    raise RuntimeError('Drive did not expose a finite storage quota; verify free space before starting.')
FREE_DRIVE_BYTES=max(0,int(quota['limit'])-int(quota.get('usage',0)))
print(f'Account free space: {FREE_DRIVE_BYTES/1e9:.2f} GB. Workflow checks the additional space needed before source reads.')
print('Destination:',OUT)

used=sum(p.stat().st_size for p in OUT.rglob('*') if p.is_file()) if OUT.exists() else 0
if FREE_DRIVE_BYTES < max(0,50_000_000-used):
    raise RuntimeError('Need up to 50 MB free for all new follow-up outputs; existing results unchanged.')


In [ ]:
import torch
if not torch.cuda.is_available():raise RuntimeError('Choose Runtime → Change runtime type → T4 GPU.')
print('GPU:',torch.cuda.get_device_name(0))


## Train and evaluate fixed grouped folds

In [ ]:
from datetime import datetime, timezone
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=[sys.executable,'-u',str(REPO/'scripts/zuco_recovered_followups.py'),
         '--thesis-root',str(THESIS),'--phase','cnn']
# Advisory status stays on runtime disk; scientific progress/results persist on Drive.
run_with_progress(command,Path('/content/zuco-followups-status-cnn.json'),activity='Recovered ZuCo cnn follow-ups on saved trials')


## CNN fold results

In [ ]:
results=pd.DataFrame(json.loads((OUT/'cnn/summary.json').read_text()))
display(results[['representation','split','fold','fit_trials','validation_trials','test_trials','epochs','accuracy','balanced_accuracy','macro_f1','auc']])
display(results.groupby(['representation','split'])[['balanced_accuracy','macro_f1','auc']].mean())
print('Fold means are descriptive; folds are not independent readers. Binary balanced-accuracy chance = 0.50.')
print('Individual fixation CNN scores should be compared to single-fixation linear scores, not 40-trial average scores.')
print('Full results:',OUT/'cnn')


## Interpretation
All results are exploratory. The original cleaning and matching used full reader recordings, so these are **transductive preprocessing** evaluations even when classifier folds hold out sentences, words or readers. They cannot establish strict fit-on-training-only preprocessing generalization. Shared words/sentences across other split types are reported. Pre-onset EEG is a context/timing control, not guaranteed sentiment-free data.

Pseudotrial accuracy uses known labels to build same-class test averages and is **not single-fixation decoding accuracy**. No neutral trials are present in this binary archive. No settings or labels are changed to make a result significant.
